In [1]:
import pandas as pd
data = pd.read_csv('../Dataset/Scored_Cleaned_Data.csv')

In [2]:
import numpy as np
import pandas as pd
from kmodes.kmodes import KModes
from sklearn.metrics import silhouette_score
from sklearn.model_selection import KFold

def evaluate_kmodes_custom_logic(df, target_col='Risk_score', n_folds=5):
    X_df = df.drop(columns=["Timestamp", "Group"], errors='ignore')
    spend_values = df[target_col].values
    
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    overall_results = []

    print(f"{'K':<5} | {'Avg Silhouette':<15} | {'Custom Deviation':<20}")
    print("-" * 50)

    for k in range(3, 20):
        fold_silhouettes = []
        fold_custom_deviations = []
        
        for train_idx, val_idx in kf.split(X_df):
            X_fold = X_df.iloc[train_idx]
            spend_fold = spend_values[train_idx]
            
            km = KModes(n_clusters=k, init='Huang', n_init=5, verbose=0, random_state=42)
            labels = km.fit_predict(X_fold)
            
            # Silhouette with hamming (correct for categorical)
            try:
                sil = silhouette_score(X_fold, labels, metric='hamming')
            except:
                sil = 0
            fold_silhouettes.append(sil)

            # (sum(|M - means|^0.5))^2
            cluster_means = [spend_fold[labels == cid].mean() for cid in np.unique(labels) if np.any(labels == cid)]
            c_means = np.array(cluster_means)
            M = np.mean(c_means)
            diffs = np.abs(M - c_means)
            custom_dev = (np.sum(diffs**0.5))**2
            fold_custom_deviations.append(custom_dev)

        avg_sil = np.mean(fold_silhouettes)
        avg_dev = np.mean(fold_custom_deviations)
        overall_results.append({'k': k, 'Silhouette': avg_sil, 'Custom_Deviation': avg_dev})
        print(f"{k:<5} | {avg_sil:<15.4f} | {avg_dev:<20.2f}")

    return pd.DataFrame(overall_results)

kmodes_custom_results = evaluate_kmodes_custom_logic(data)

K     | Avg Silhouette  | Custom Deviation    
--------------------------------------------------
3     | 0.0897          | 173.76              
4     | 0.0829          | 269.94              
5     | 0.0872          | 440.33              
6     | 0.0946          | 611.01              
7     | 0.0951          | 888.08              
8     | 0.1049          | 1179.95             
9     | 0.1004          | 1456.98             
10    | 0.1190          | 1862.07             
11    | 0.1140          | 2213.43             
12    | 0.1216          | 2564.58             
13    | 0.1197          | 2881.64             
14    | 0.1183          | 3462.66             
15    | 0.1198          | 4318.13             
16    | 0.1283          | 4453.82             
17    | 0.1385          | 5661.26             
18    | 0.1514          | 5977.04             
19    | 0.1429          | 7056.74             
